# RPDI-EE trên vLLM — chạy thử 200 câu (ids 100–299) ở ngân sách 16.384 token

Mục tiêu: có số liệu **cùng mô hình, cùng prompt, cùng tập câu** để so RPDI-EE ở 768 token với thiết lập của bài gốc (16.384 token, W=512, λ=2,0).

Mỗi cặp (câu, phương pháp) chỉ sinh **một lần** ở 16.384 token. Kết quả ở 512 / 768 / 1.024 / … / 8.192 token được suy ra bằng cách cắt chuỗi token. Cách này hợp lệ vì giải mã greedy và quyết định phanh chỉ phụ thuộc phần đã sinh.

| Phương pháp | Ý nghĩa |
|---|---|
| `standard` | Vanilla, không phanh (vẫn ghi entropy để phân tích) |
| `rpdi_legacy:16:1.5` | Bản sao chính xác cài đặt HF đã cho kết quả 71,5% ở 768 token |
| `rpdi:512:2.0` | Đúng Algorithm 1 của bài gốc, bộ biên = "Wait" (điểm chuyển ý của DEER) |
| `rpdi:512:2.0:newline` | Như trên nhưng bộ biên = token xuống dòng (cách hiểu thứ hai của "giống DEER") |

**Trước khi chạy:** code mới phải đã được push lên GitHub, vì notebook clone repo từ đó.

**Thứ tự:** chạy lần lượt từ trên xuống. Cell 5 (smoke test) mất khoảng 5–10 phút; nếu nó báo lỗi, dừng lại và gửi lỗi cho Claude. Cell 6 chạy nền; thời gian ước tính 30–90 phút, xem ETA ở cell 7.

## 1. Kiểm tra GPU và môi trường

In [ ]:
import os, subprocess
# Không dựa vào thư mục /kaggle: máy Colab cũng có sẵn thư mục này
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print("Môi trường:", "Kaggle" if IN_KAGGLE else "Colab")
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print(gpus or "KHÔNG THẤY GPU")
if not gpus:
    raise RuntimeError("Chưa bật GPU. Colab: Runtime → Change runtime type → T4 GPU. Kaggle: Settings → Accelerator → GPU T4 x2.")
if "P100" in gpus:
    raise RuntimeError("vLLM không chạy trên P100. Kaggle: Settings → Accelerator → GPU T4 x2.")

## 2. Lấy code mới nhất

In [ ]:
REPO_DIR = "/kaggle/working/slm-reasoning-project" if IN_KAGGLE else "/content/slm-reasoning-project"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/luw-cid/slm-reasoning-project.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git log --oneline -1

## 3. Cài vLLM (bản V0, có logits processor theo từng request)

vLLM 0.6.3 chỉ hỗ trợ Python ≤ 3.12, nên cell này tạo một môi trường Python 3.12 riêng ở `/tmp/vllm-env` và cho các lệnh `!python` ở những cell sau dùng môi trường đó. Môi trường có sẵn của Colab/Kaggle không bị đụng tới, nên **không cần khởi động lại runtime**. Nếu kernel bị khởi động lại, chỉ cần chạy lại cell này.

In [ ]:
# vLLM 0.6.3 (engine V0, có logits processor theo từng request) chỉ chạy trên Python <= 3.12,
# trong khi Kaggle/Colab có thể đã lên Python 3.13. Tạo môi trường Python 3.12 riêng bằng uv.
import os, subprocess
VENV = "/tmp/vllm-env"   # ngoài /kaggle/working để không bị lưu vào output của Kaggle
PYBIN = f"{VENV}/bin/python"
CHECK = ("import vllm, torch, transformers; print('vllm', vllm.__version__, '| torch', torch.__version__, "
         "'| transformers', transformers.__version__, '| CUDA', torch.cuda.is_available())")

if not (os.path.exists(PYBIN) and subprocess.run([PYBIN, "-c", "import vllm"], capture_output=True).returncode == 0):
    print("Đang cài môi trường Python 3.12 + vLLM 0.6.3 (khoảng 3–6 phút)...")
    !pip install -q uv
    !python -m uv venv {VENV} --python 3.12
    !python -m uv pip install --python {PYBIN} "vllm==0.6.3.post1" "transformers==4.46.3" pytest 2>&1 | tail -n 15

# Từ đây, mọi lệnh `!python ...` ở các cell sau sẽ dùng Python của môi trường này.
# Nếu kernel bị khởi động lại, chạy lại cell này (lần sau chỉ mất vài giây).
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
check = subprocess.run(["python", "-c", CHECK], capture_output=True, text=True)
print(check.stdout or check.stderr[-3000:])
if check.returncode != 0:
    raise RuntimeError("Cài vLLM thất bại. Gửi toàn bộ output của cell này cho Claude.")
!python -m pytest tests -q

## 4. Thư mục kết quả và biên δ

**δ phải được chốt với thầy hướng dẫn trước khi xem bảng so sánh ở cell 8**, và không sửa sau khi đã thấy kết quả. Nếu chưa chốt, đặt `DELTA = None`: cell 8 vẫn in bảng theo ngân sách và phần kiểm tra pipeline, nhưng bỏ qua phần kết luận.

In [ ]:
if IN_KAGGLE:
    OUT_ROOT = "/kaggle/working/vllm_runs"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = "/content/drive/MyDrive/slm_rlvr/vllm_runs"

SMOKE_DIR = f"{OUT_ROOT}/smoke"
PILOT_DIR = f"{OUT_ROOT}/pilot_200"
METHODS = "standard,rpdi_legacy:16:1.5,rpdi:512:2.0,rpdi:512:2.0:newline"
HF_REF = "results/rpdi_results/controls_standard.jsonl results/rpdi_results/controls_rpdi.jsonl"
VALIDATE = "standard=standard rpdi_legacy_W16_l1.5=rpdi_W16_l1.5"

DELTA = 3.0  # điểm %; ĐỔI theo con số đã chốt với thầy (hoặc None nếu chưa chốt)

os.makedirs(PILOT_DIR, exist_ok=True)
print("Kết quả sẽ lưu tại:", OUT_ROOT)

## 5. Smoke test: 3 câu (ids 100–102), ngân sách 1.024

Kiểm tra: vLLM chạy được trên T4, entropy là số hữu hạn, RPDI-EE có phanh. Bảng cuối so với kết quả HF đã có của chính 3 câu này. Lý tưởng là "Trùng văn bản" bằng 100%; thấp hơn một chút vẫn chấp nhận được (fp16 trên hai engine khác nhau), quan trọng là accuracy ở cell 8.

In [ ]:
import json, math
!rm -rf {SMOKE_DIR}
!python -u method_c_rpdi_ee/vllm_engine/run_vllm.py --start 100 --end 103 --max_tokens 1024 --budgets 512,768,1024 --methods standard,rpdi_legacy:16:1.5,rpdi:512:2.0 --chunk_size 3 --output_dir {SMOKE_DIR}

raws = [json.loads(l) for l in open(f"{SMOKE_DIR}/raw.jsonl", encoding="utf-8")]
assert len(raws) == 9, f"Cần 9 dòng, có {len(raws)}"
for r in raws:
    assert r["entropies"] and all(math.isfinite(h) for h in r["entropies"]), (r["sample_id"], r["method"])
    print(f"id={r['sample_id']} {r['method']:<22} token={len(r['output_ids']):>5} </think>@{r['close_index']} "
          f"phanh={r['forced_close']} exit_step={r['exit_step']} H[:3]={r['entropies'][:3]}")
if not any(r["forced_close"] for r in raws if r["method"].startswith("rpdi_legacy")):
    print("CẢNH BÁO: rpdi_legacy không phanh câu nào trong 3 câu — gửi output này cho Claude trước khi chạy tiếp.")
!python method_c_rpdi_ee/vllm_engine/analyze_budgets.py --budgets_file {SMOKE_DIR}/budgets.jsonl --hf_reference {HF_REF} --validate {VALIDATE} --n_boot 200 > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{SMOKE_DIR}/analysis.md", encoding="utf-8").read()))

## 6. Chạy thử 200 câu ở 16.384 token (chạy nền)

Chạy bằng `nohup` nên đóng tab vẫn không mất tiến trình, miễn là runtime còn sống. Nếu runtime bị ngắt: chạy lại các cell 1–4 rồi cell này; script tự bỏ qua các cặp đã xong (lưu theo từng chunk 50 câu).

In [ ]:
LOG = f"{PILOT_DIR}/log.txt"
cmd = (f"nohup python -u method_c_rpdi_ee/vllm_engine/run_vllm.py --start 100 --end 300 --max_tokens 16384 "
       f"--methods {METHODS} --chunk_size 50 --output_dir {PILOT_DIR} >> {LOG} 2>&1 &")
print(cmd)
os.system(cmd)

## 7. Theo dõi tiến độ (chạy lại cell này bao nhiêu lần cũng được)

In [ ]:
!tail -c 3000 {LOG} | tr '\r' '\n' | grep -v '^$' | tail -n 8
!echo "Số dòng raw (cần 800 = 200 câu x 4 phương pháp):" && (wc -l < {PILOT_DIR}/raw.jsonl 2>/dev/null || echo 0)
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

## 8. Phân tích (khi đủ 800 dòng)

- **Kiểm tra pipeline:** `standard` và `rpdi_legacy_W16_l1.5` ở 768 token phải lệch không quá 2 điểm so với HF (52,5% và 71,5%) thì mới tin được các con số 16k.
- **So sánh chính:** RPDI-EE 768 token của bạn so với thiết lập bài gốc ở 16.384 token (hai cách hiểu bộ biên) và so với vanilla 16.384.

In [ ]:
COMPARE = ("rpdi_legacy_W16_l1.5@768:rpdi_W512_l2.0_wait@16384 "
           "rpdi_legacy_W16_l1.5@768:rpdi_W512_l2.0_newline@16384 "
           "rpdi_legacy_W16_l1.5@768:standard@16384")
DELTA_ARG = f"--delta {DELTA}" if DELTA is not None else ""
!python method_c_rpdi_ee/vllm_engine/analyze_budgets.py --budgets_file {PILOT_DIR}/budgets.jsonl --hf_reference {HF_REF} --validate {VALIDATE} --compare {COMPARE} {DELTA_ARG} > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{PILOT_DIR}/analysis.md", encoding="utf-8").read()))

## 9. Gửi lại kết quả

Tải từ Drive (`MyDrive/slm_rlvr/vllm_runs/pilot_200/`) các file sau và chép vào `results/vllm_pilot_200/` trong repo:

- `analysis.md`, `analysis.json`: bảng kết quả
- `budgets.jsonl`: kết quả từng câu ở từng ngân sách
- `run_info.json`, `log.txt`: phiên bản thư viện và thời gian chạy

`raw.jsonl` khá lớn (chứa toàn bộ token và entropy), cứ để trên Drive; chỉ cần khi muốn phân tích lại.